## Taller "Diseño e implementación de flujos de trabajo agénticos"
### I Congreso de Inteligencia Artificial, UNAH
### Tegucigalpa, Honduras, 13-17 julio de 2026 

## RAG (Retrieval-augmented generation)

RAG es una arquitectura que permite mejorar la respuesta de un modelo de lenguaje, que se basa en brindar a un modelo de lenguaje acceso a información adicional, de modo que el contexto de las peticiones de usuario sean enriquecidas. 

La técnica de RAG se basa en el acceso (retrieval) a una **base de datos vectorizada (vector database)**. Esta consiste de texto (documentos) almacenado en un formato que los representa como vectores. Esto permite una **búsqueda por similitud** que permite convertir la noción de similitud semántica en cercanía de vectores en el sentido geométrico. 

  La creación de una base de dactos vectorizada sigue los siguientes pasos.

-*Chunking:* El texto de cada documento se divide en fragmentos pequeños (chunks).

-El archivo de chunks es consumido por un modelo de embedding.

-El modelo de embedding mapea cada chunk a una representación vectorial. 

-Los vectores son almacenados en una base de datos vectorizada.

Una vez disponible una base de datos vectorizada, un LLM+RAG funciona con los siguientes pasos.

-El usuario envía su petición al LLM.

-La petición es pasada al modelo de embedding.

-*Retrieval:* Dentro de la base de datos vectorizada, se efectúa una búsqueda de similitud de la petición.

-Los resultados más cercanos (contexto) se agregan a la petición original del usuario.

-El LLM consume la petición más el contexto y produce un output. 

## Ejemplo de implementación 

En este ejemplo, construiremos un flujo de RAG a partir del texto del Código Penal de El Salvador.

### Instalación de dependencias

In [ ]:
!pip install langchain_chroma -q

In [ ]:
!pip install langchain_huggingface -q

In [ ]:
!pip install langchain_text_splitters -q

In [ ]:
!pip install langchain_openai -q 

In [ ]:
#Dependencias
from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI


### Variables

In [ ]:
#Llave de API Groq
API_KEY = 


In [ ]:
# Celda para carga de archivo TXT en Google Colab
from google.colab import files

_ = files.upload()

In [ ]:
DOCUMENT_PATH = "normalized-codigo-penal.txt"
VECTORSTORE_PATH = "./chroma_db"
COLLECTION_NAME = "codigo_penal"


### Definiciones

In [ ]:
#Carga de archivo de texto como objeto Document

def load_document(file_path: str) -> list[Document]:
    text = Path(file_path).read_text(encoding="utf-8")

    return [
        Document(
            page_content=text,
            metadata={"source": file_path},
        )
    ]


In [ ]:
#Función chunking

def split_documents(documents: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
    )

    return splitter.split_documents(documents)



In [ ]:
#Carga de embedding model

def create_embeddings() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        encode_kwargs={"normalize_embeddings": True},
    )



In [ ]:
#Creación de base de datos vectorizada (ChromaDB)

def create_vectorstore(
    chunks: list[Document],
    embeddings: HuggingFaceEmbeddings,
) -> Chroma:
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=VECTORSTORE_PATH,
    )

    vectorstore.add_documents(chunks)

    return vectorstore



In [ ]:
#Función de retrieval

def retrieve_context(
    vectorstore: Chroma,
    query: str,
    top_k: int = 5,
) -> list[Document]:
    return vectorstore.similarity_search(
        query=query,
        k=top_k,
    )



In [ ]:
#Llamada a LLM 

def generate_answer(
    query: str,
    retrieved_documents: list[Document],
) -> str:
    context = "\n\n---\n\n".join(
        document.page_content
        for document in retrieved_documents
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
Responde la pregunta usando únicamente el contexto indicado.

Si la respuesta no está en el contexto, responde: NO LO SÉ.
""",
            ),
            (
                "human",
                """
Contexto:

{context}

Pregunta:

{question}
""",
            ),
        ]
    )

    #Llamada a modelo mediante Groq
    llm = ChatOpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
    model="llama-3.3-70b-versatile", 
    temperature=0.0 # Temperatura baja para retrieval
)

    chain = prompt | llm

    response = chain.invoke(
        {
            "context": context,
            "question": query,
        }
    )

    return response.content



### Ejecución del flujo

In [ ]:
# Carga de documento
documents = load_document(DOCUMENT_PATH)


In [ ]:
# Función chunking (CORRER SÓLO UNA VEZ O HABRÁ CHUNKS REPETIDOS!)
chunks = split_documents(documents)


In [ ]:
# Carga de embedding model
embeddings = create_embeddings()


In [ ]:
# Creación o carga de base de datos vectorizada
# Si ya fue creada (revisar directorio /chromadb) no es necesario repetir
vectorstore = create_vectorstore(
    chunks=chunks,
    embeddings=embeddings,
)


In [ ]:
# Query de usuario 
question = input("Question: ")


In [ ]:
question = '¿Cuál es la pena por homicidio simple?'

In [ ]:
# Consulta (retrieval) de documentos
retrieved_documents = retrieve_context(
    vectorstore=vectorstore,
    query=question,
    top_k=5,
)


In [ ]:
# Generación de respuesta (llamada a modelo de lenguaje)
answer = generate_answer(
    query=question,
    retrieved_documents=retrieved_documents,
)


In [ ]:
#Output
print("\n Respuesta:")
print(answer)


In [ ]:
#Inspección de chunks
print("\n Chunks extraídos:")

for index, document in enumerate(retrieved_documents, start=1):
    print(f"\n--- Chunk {index} ---")
    print(document.page_content[:500])

### Observaciones

En este flujo básico, podemos notar la baja calidad de las respuestas. Los siguientes aspectos son clave.

-El modelo de embedding debe acoplarse al lenguaje (en nuestro caso, español).

-La función chunking debe adaptarse a la estructura del texto (en este caso, un documento legal).